<a href="https://colab.research.google.com/github/caochengrui/DQN/blob/main/DQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Network (DQN)



### Install Dependencies

In [1]:
!pip install git+https://github.com/caochengrui/test --upgrade

  Cloning https://github.com/caochengrui/test to /tmp/pip-req-build-zelk5w8r
  Running command git clone --filter=blob:none --quiet https://github.com/caochengrui/test /tmp/pip-req-build-zelk5w8r
  Resolved https://github.com/caochengrui/test to commit 3fc3b3b31a98012af2d3d8c5a7cf23d00181fbc7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 45.1 MB/s eta 0:00:00
  Created wheel for DQN: filename=dqn-0.1.dev1+g3fc3b3b31-py3-none-any.whl size=26167 sha256=5bba4088cd97aba88371c56712abde39507716b9c325b7d66042bc8976f6a813
  Stored in directory: /tmp/pip-ephem-wheel-cache-g0d_h755/wheels/12/c7/c2/ee15cc7997af7d63a89c08c5951cae695894514daa92a96f71
Successfully built DQN
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


In [2]:
!apt-get install -y ffmpeg  # For visualization
!pip install opencv-python  # For image preprocessing in visual DQN

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


### Imports

In [3]:
from typing import Optional
import os

import numpy as np
import torch as th
import gymnasium as gym
from gymnasium import spaces

from DQN import (
    ReplayBuffer,
    epsilon_greedy_action_selection,
    collect_one_step,
    linear_schedule,
    QNetwork,
    CNNQNetwork,
    make_flappy_env,
)
from DQN.evaluation import evaluate_policy
from custom_utils import notebook_show_videos

##  DQN Target Network


<div>
    <img src="attachment:c176b66c-2fc7-4e1e-a351-87ef832bdfac.png" width="1000"/>
</div>

The only things that is changing is when predicting the next q value.

In DQN without target, the online network with weights **$\theta$** is used:

$y = r_t + \gamma \cdot \max_{a \in A}(\hat{Q}_{\pi}(s_{t+1}, a; \theta))$


whereas with DQN with target network, the target q-network (a delayed copy of the q-network) with weights **$\theta^\prime$** is used instead:

$y = r_t + \gamma \cdot \max_{a \in A}(\hat{Q}_{\pi}(s_{t+1}, a; \theta^\prime))$


### DQN update with target network

In [4]:
def dqn_update(
    q_net: QNetwork,
    q_target_net: QNetwork,
    optimizer: th.optim.Optimizer,
    replay_buffer: ReplayBuffer,
    batch_size: int,
    gamma: float,
) -> None:
    """
    Perform one gradient step on the Q-network
    using the data from the replay buffer.

    :param q_net: The Q-network to update
    :param q_target_net: The target Q-network, to compute the td-target.
    :param optimizer: The optimizer to use
    :param replay_buffer: The replay buffer containing the transitions
    :param batch_size: The minibatch size, how many transitions to sample
    :param gamma: The discount factor
    """

    replay_data = replay_buffer.sample(batch_size).to_torch()

    with th.no_grad():
        next_q_values = q_target_net(replay_data.next_observations)
        next_q_values, _ = next_q_values.max(dim=1)
        # If the episode is terminated, set the target to the reward
        should_bootstrap = th.logical_not(replay_data.terminateds)
        td_target = replay_data.rewards + gamma * next_q_values * should_bootstrap

    q_values = q_net(replay_data.observations)
    # Select the Q-values corresponding to the actions that were selected
    # during data collection
    current_q_values = th.gather(q_values, dim=1, index=replay_data.actions)
    # Reshape from (batch_size, 1) to (batch_size,) to avoid broadcast error
    current_q_values = current_q_values.squeeze(dim=1)

    assert current_q_values.shape == (batch_size,), f"{current_q_values.shape} != {(batch_size,)}"
    assert current_q_values.shape == td_target.shape, f"{current_q_values.shape} != {td_target.shape}"

    # Compute the Mean Squared Error (MSE) loss
    # Optionally, one can use a Huber loss instead of the MSE loss
    loss = ((current_q_values - td_target) ** 2).mean()
    # Huber loss
    # loss = th.nn.functional.smooth_l1_loss(current_q_values, td_target)


    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

### Updated training loop

In [5]:
def run_dqn(
    env_id: str = "CartPole-v1",
    replay_buffer_size: int = 50_000,
    target_network_update_interval: int = 1000,
    # Warmup phase
    learning_starts: int = 100,
    exploration_initial_eps: float = 1.0,
    exploration_final_eps: float = 0.01,
    exploration_fraction: float = 0.1,
    n_timesteps: int = 20_000,
    update_interval: int = 2,
    learning_rate: float = 3e-4,
    batch_size: int = 64,
    gamma: float = 0.99,
    n_hidden_units: int = 64,
    n_eval_episodes: int = 10,
    evaluation_interval: int = 1000,
    eval_exploration_rate: float = 0.0,
    seed: int = 2026,
    eval_render_mode: Optional[str] = None,  # "human", "rgb_array", None
) -> QNetwork:
    """
    Run Deep Q-Learning (DQN) on a given environment.
    (with a target network)

    :param env_id: Name of the environment
    :param replay_buffer_size: Max capacity of the replay buffer
    :param target_network_update_interval: How often do we copy the parameters
         to the target network
    :param learning_starts: Warmup phase to fill the replay buffer
        before starting the optimization.
    :param exploration_initial_eps: The initial exploration rate
    :param exploration_final_eps: The final exploration rate
    :param exploration_fraction: The fraction of the number of steps
        during which the exploration rate is annealed from
        initial_eps to final_eps.
        After this many steps, the exploration rate remains constant.
    :param n_timesteps: Number of timesteps in total
    :param update_interval: How often to update the Q-network
        (every update_interval steps)
    :param learning_rate: The learning rate to use for the optimizer
    :param batch_size: The minibatch size
    :param gamma: The discount factor
    :param n_hidden_units: Number of units for each hidden layer
        of the Q-Network.
    :param n_eval_episodes: The number of episodes to evaluate the policy on
    :param evaluation_interval: How often to evaluate the policy
    :param eval_exploration_rate: The exploration rate to use during evaluation
    :param seed: Random seed for the pseudo random generator
    :param eval_render_mode: The render mode to use for evaluation
    """

    np.random.seed(seed)
    th.manual_seed(seed)

    os.makedirs("./logs/", exist_ok=True)
    os.makedirs("./logs/checkpoint/", exist_ok=True)

    env = gym.make(env_id)
    # For highway env
    env = gym.wrappers.FlattenObservation(env)
    env = gym.wrappers.RecordEpisodeStatistics(env)
    assert isinstance(env.observation_space, spaces.Box)
    assert isinstance(env.action_space, spaces.Discrete)
    env.action_space.seed(seed)

    eval_env = gym.make(env_id, render_mode=eval_render_mode)
    eval_env = gym.wrappers.FlattenObservation(eval_env)
    eval_env.reset(seed=seed)
    eval_env.action_space.seed(seed)


    q_net = QNetwork(env.observation_space, env.action_space, n_hidden_units=n_hidden_units)
    q_target_net = QNetwork(env.observation_space, env.action_space, n_hidden_units=n_hidden_units)
    q_target_net.load_state_dict(q_net.state_dict())

    # For flappy bird
    if env.observation_space.dtype == np.float64:
        q_net.double()
        q_target_net.double()

    optimizer = th.optim.Adam(q_net.parameters(), lr=learning_rate)

    replay_buffer = ReplayBuffer(replay_buffer_size, env.observation_space, env.action_space)
    obs, _ = env.reset(seed=seed)
    for current_step in range(1, n_timesteps + 1):
        exploration_rate = linear_schedule(
            exploration_initial_eps,
            exploration_final_eps,
            current_step,
            int(exploration_fraction * n_timesteps),
        )

        obs = collect_one_step(
            env,
            q_net,
            replay_buffer,
            obs,
            exploration_rate=exploration_rate,
            verbose=0,
        )

        if (current_step % target_network_update_interval) == 0:
            q_target_net.load_state_dict(q_net.state_dict())

        if (current_step % update_interval) == 0 and current_step > learning_starts:
            dqn_update(q_net, q_target_net, optimizer, replay_buffer, batch_size, gamma=gamma)

        if (current_step % evaluation_interval) == 0:
            print()
            print(f"Evaluation at step {current_step}:")
            evaluate_policy(eval_env, q_net, n_eval_episodes, eval_exploration_rate=eval_exploration_rate)
            th.save(q_net.state_dict(), f"./logs/checkpoint/q_net_checkpoint_{env_id}_{current_step}.pth")
    return q_net

## Train DQN agent with target network on CartPole env

In [6]:
# Tuned hyperparameters from the RL Zoo3 of the Stable Baselines3 library
# https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml

env_id = "CartPole-v1"

q_net = run_dqn(
    env_id=env_id,
    replay_buffer_size=100_000,
    target_network_update_interval=10,
    learning_starts=1000,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.04,
    exploration_fraction=0.1,
    n_timesteps=80_000,
    update_interval=2,
    learning_rate=1e-3,
    batch_size=64,
    gamma=0.99,
    n_eval_episodes=10,
    evaluation_interval=5000,
    eval_exploration_rate=0.0,
    seed=2026,
)


Evaluation at step 5000:
Mean episode reward: 236.20 +/- 65.39

Evaluation at step 10000:
Mean episode reward: 268.80 +/- 110.91

Evaluation at step 15000:
Mean episode reward: 251.80 +/- 74.17

Evaluation at step 20000:
Mean episode reward: 480.30 +/- 59.10

Evaluation at step 25000:
Mean episode reward: 334.00 +/- 109.74

Evaluation at step 30000:
Mean episode reward: 487.20 +/- 38.40

Evaluation at step 35000:
Mean episode reward: 432.10 +/- 100.33

Evaluation at step 40000:
Mean episode reward: 332.50 +/- 44.06

Evaluation at step 45000:
Mean episode reward: 255.30 +/- 21.19

Evaluation at step 50000:
Mean episode reward: 247.30 +/- 34.22

Evaluation at step 55000:
Mean episode reward: 211.70 +/- 10.81

Evaluation at step 60000:
Mean episode reward: 217.70 +/- 12.88

Evaluation at step 65000:
Mean episode reward: 478.00 +/- 39.90

Evaluation at step 70000:
Mean episode reward: 356.80 +/- 21.12

Evaluation at step 75000:
Mean episode reward: 500.00 +/- 0.00

Evaluation at step 8000

### Visualize the trained agent

In [7]:
eval_env = gym.make(env_id, render_mode="rgb_array")
n_eval_episodes = 3
eval_exploration_rate = 0.0
video_name = f"DQN_{env_id}"

q_net = QNetwork(eval_env.observation_space, eval_env.action_space, n_hidden_units=64)
q_net.load_state_dict(th.load("./logs/checkpoint/q_net_checkpoint_CartPole-v1_75000.pth"))

evaluate_policy(
    eval_env,
    q_net,
    n_eval_episodes,
    eval_exploration_rate=eval_exploration_rate,
    video_name=video_name,
)

notebook_show_videos("./logs/videos/", prefix=video_name)

Saving video to logs/videos/DQN_CartPole-v1.mp4
Mean episode reward: 500.00 +/- 0.00


## Training DQN agent on flappy bird:

You can go in the [GitHub repo](https://github.com/araffin/flappy-bird-gymnasium/tree/patch-1) to learn more about this environment.

<div>
    <img src="https://raw.githubusercontent.com/markub3327/flappy-bird-gymnasium/main/imgs/dqn.gif" width="300"/>
</div>


In [8]:
!pip install "flappy-bird-gymnasium @ git+https://github.com/araffin/flappy-bird-gymnasium@patch-1"

  Cloning https://github.com/araffin/flappy-bird-gymnasium (to revision patch-1) to /tmp/pip-install-upi32hpz/flappy-bird-gymnasium_2dd5c4df46b54ed9b8e2783fe37e6f05
  Running command git clone --filter=blob:none --quiet https://github.com/araffin/flappy-bird-gymnasium /tmp/pip-install-upi32hpz/flappy-bird-gymnasium_2dd5c4df46b54ed9b8e2783fe37e6f05
  Running command git checkout -b patch-1 --track origin/patch-1
  Switched to a new branch 'patch-1'
  Branch 'patch-1' set up to track remote branch 'patch-1' from 'origin'.
  Resolved https://github.com/araffin/flappy-bird-gymnasium to commit 8828737242a38c0cd18c4260819fe7e695fb6bae
  Preparing metadata (setup.py) ... done
  Created wheel for flappy-bird-gymnasium: filename=flappy_bird_gymnasium-0.2.2-py3-none-any.whl size=1074471 sha256=f6a95ea5bd0249b23b0f17d89afd6fe44e5ebd8262fe6ae9e7a7fae3622a4449
  Stored in directory: /tmp/pip-ephem-wheel-cache-x5jfhs41/wheels/b0/48/e5/e447569a935d86b16cf1fd7560c83ae96ea0ebaaec403a04f2
Successfully b

In [9]:
import flappy_bird_gymnasium  # noqa: F401

In [10]:
env_id = "FlappyBird-v0"

q_net = run_dqn(
    env_id=env_id,
    replay_buffer_size=100_000,
    target_network_update_interval=250,
    learning_starts=10_000,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.03,
    exploration_fraction=0.1,
    n_timesteps=500_000,
    update_interval=4,
    learning_rate=1e-3,
    batch_size=128,
    gamma=0.98,
    n_eval_episodes=5,
    evaluation_interval=50000,
    n_hidden_units=256,
    eval_exploration_rate=0.0,
    seed=2026,
    eval_render_mode=None,
)


Evaluation at step 50000:
Mean episode reward: 9.14 +/- 0.17

Evaluation at step 100000:
Mean episode reward: 14.40 +/- 1.80

Evaluation at step 150000:
Mean episode reward: 17.24 +/- 5.41

Evaluation at step 200000:
Mean episode reward: 49.80 +/- 48.23

Evaluation at step 250000:
Mean episode reward: 152.16 +/- 108.85

Evaluation at step 300000:
Mean episode reward: 213.20 +/- 141.06

Evaluation at step 350000:
Mean episode reward: 923.40 +/- 1116.54

Evaluation at step 400000:
Mean episode reward: 111.12 +/- 101.24

Evaluation at step 450000:
Mean episode reward: 1397.28 +/- 1816.98

Evaluation at step 500000:
Mean episode reward: 481.90 +/- 377.90


### Record a video of the trained agent

In [11]:
eval_env = gym.make(env_id, render_mode="rgb_array")
n_eval_episodes = 3
eval_exploration_rate = 0.00
video_name = f"DQN_{env_id}"
q_net = QNetwork(eval_env.observation_space, eval_env.action_space, n_hidden_units=256)
# Convert weights from float32 to float64 to match flappy bird obs
q_net.double()
q_net.load_state_dict(th.load("./logs/checkpoint/q_net_checkpoint_FlappyBird-v0_500000.pth"))

evaluate_policy(
    eval_env,
    q_net,
    n_eval_episodes,
    eval_exploration_rate=eval_exploration_rate,
    video_name=video_name,
)

notebook_show_videos("./logs/videos/", prefix=video_name)

Saving video to logs/videos/DQN_FlappyBird-v0.mp4
Mean episode reward: 128.50 +/- 82.12


---

## Visual DQN on Flappy Bird (learning from pixels)

The earlier "visual DQN" was trained on **CartPole rendered to pixels** — but CartPole's vector state (cart position, pole angle, velocities) is *exactly* what makes it learnable, so throwing it away to learn from a rendered image is a losing trade. The natural task for a pixel-based agent is a game whose native interface **is** the screen: **Flappy Bird**.

This section trains a CNN-based DQN directly on the RGB game screen of `FlappyBird-rgb-v0`. It is a **full rewrite** of the visual pipeline. The previous version lost badly to the vector-observation DQN because it reused a training loop and hyperparameters tuned for vector observations. The rewrite (in `DQN/train_flappy.py`) adds the ingredients pixel-based DQN actually needs:

- **Double DQN** target — removes the Q-value overestimation bias that destabilises pixel training
- **Huber loss** + **gradient-norm clipping** — robust to the large TD errors common early on
- **Reward clipping** — keeps the TD-target scale stable
- **Warmup + `train_freq`** — one gradient step every few env steps, *after* a warmup phase (not an update on almost every step)
- **Long ε-annealing** over a much larger training budget
- **uint8 replay buffer** with GPU-side normalisation, so a large image buffer fits in RAM

**Preprocessing pipeline** (`make_flappy_env`):

`RGB screen → grayscale → resize 84×84 → stack 4 frames → (4, 84, 84)`

**CNN architecture** (`CNNQNetwork`, Nature-DQN):
Conv 8×8/4 → Conv 4×4/2 → Conv 3×3/1 → FC 512 → `n_actions`, with orthogonal init.

> Requires `flappy-bird-gymnasium` — installed by the cell near the start of the Flappy Bird section above:
> `pip install "flappy-bird-gymnasium @ git+https://github.com/araffin/flappy-bird-gymnasium@patch-1"`

### Train the Visual DQN agent

`train()` runs the whole tuned loop — data collection, Double-DQN updates, target-network syncs, periodic evaluation and checkpointing — and returns the trained network. All time-based arguments are counted in **environment steps**.

`total_timesteps` is the main quality/time knob. The default-style 1M agent steps (2M game frames at `frame_skip=2`) was verified on an L4 GPU: it takes roughly **45–60 min** there (longer on a slower Colab GPU) and lifts the pixel agent from the ~9 untrained baseline to a **mean eval return of ~130** — well above the vector-observation DQN trained on `FlappyBird-v0` earlier. Reduce it for a quick run, increase it for a stronger agent. Checkpoints go to `logs/checkpoint/` (including `visual_dqn_flappy_best.pt`, the best-by-evaluation snapshot).

> Quick pipeline check before committing to a full run: `!python -m DQN.train_flappy --smoke`

In [12]:
from DQN.train_flappy import train, record_video

# This configuration was verified end-to-end on an L4 GPU: it took the pixel
# agent from a ~9 untrained baseline to a mean return of ~130 over 30 eval
# episodes (vs ~35 for the vector-observation DQN on FlappyBird-v0 above).
visual_q_net = train(
    env_id="FlappyBird-rgb-v0",
    total_timesteps=1_000_000,    # 2M game frames at frame_skip=2; ~45-60 min on an L4 GPU
    buffer_size=100_000,          # ~5.6 GB RAM at (4, 84, 84) uint8 — raise it if RAM allows
    learning_starts=20_000,       # warmup steps before the first gradient update
    batch_size=32,
    learning_rate=1e-4,
    gamma=0.99,
    train_freq=4,                 # one gradient step per 4 env steps
    target_update_interval=2_000,
    exploration_fraction=0.1,     # anneal epsilon over the first 10% of training
    exploration_final_eps=0.01,
    double_dqn=True,
    frame_skip=2,                 # repeat each action over 2 game frames (verified)
    frame_stack=4,
    grayscale=True,
    eval_freq=100_000,
    n_eval_episodes=5,
    seed=2026,
    device="auto",                # CUDA -> Apple MPS -> CPU
)

[train] device: cuda | cpu threads: 8
[train] observation space: (4, 84, 84) uint8 | action space: 2
[train] CNN parameters: 1,685,154 | replay buffer ~5.64 GB (100,000 transitions)
[train] step     2,000/1,000,000 | eps 0.980 | ep_return    9.00 | ep_len     51 | updates        0 |   225 step/s
[train] step     4,000/1,000,000 | eps 0.960 | ep_return    9.00 | ep_len     51 | updates        0 |   235 step/s
[train] step     6,000/1,000,000 | eps 0.941 | ep_return    9.00 | ep_len     51 | updates        0 |   238 step/s
[train] step     8,000/1,000,000 | eps 0.921 | ep_return    9.00 | ep_len     51 | updates        0 |   240 step/s
[train] step    10,000/1,000,000 | eps 0.901 | ep_return    9.00 | ep_len     51 | updates        0 |   240 step/s
[train] step    12,000/1,000,000 | eps 0.881 | ep_return    9.00 | ep_len     51 | updates        0 |   240 step/s
[train] step    14,000/1,000,000 | eps 0.861 | ep_return    9.00 | ep_len     51 | updates        0 |   240 step/s
[train] step 

### Record and visualize the trained agent

`record_video` rolls out the trained policy and saves MP4s of the **full-resolution game screen** (not the 84×84 grayscale stack the network actually sees). The agent acts on the preprocessed observations, while each video frame comes from `env.render()`.

To watch the best-by-evaluation checkpoint instead of the final weights, load `logs/checkpoint/visual_dqn_flappy_best.pt` into a fresh `CNNQNetwork` first (see the commented lines below).

In [13]:
video_name = "visual_dqn_flappy"

# To visualize the best checkpoint instead of the final weights, uncomment:
# tmp_env = make_flappy_env("FlappyBird-rgb-v0", frame_stack=4, grayscale=True)
# visual_q_net = CNNQNetwork(tmp_env.observation_space, tmp_env.action_space)
# visual_q_net.load_state_dict(th.load("./logs/checkpoint/visual_dqn_flappy_best.pt"))
# tmp_env.close()

record_video(
    visual_q_net,
    env_id="FlappyBird-rgb-v0",
    video_folder="./logs/videos",
    name_prefix=video_name,
    n_episodes=3,
    frame_skip=2,      # must match the training preprocessing
    frame_stack=4,
    grayscale=True,
    device="auto",
)

notebook_show_videos("./logs/videos/", prefix=video_name)

[video] saved 3 episode(s) to './logs/videos/' (prefix 'visual_dqn_flappy')
[video] return 40.83 +/- 15.13


### Going further

- **Compare** the Visual DQN's score against the vector-observation DQN trained on `FlappyBird-v0` earlier in this notebook — the vector agent sees exact distances and velocities, so matching it from raw pixels is the real challenge
- Tune `frame_skip` (1 vs 2 vs 4) and `total_timesteps` on a GPU box
- Ablate `frame_stack`, or try RGB input (`grayscale=False`) instead of grayscale
- Analyse the learned Q-values; experiment with soft (Polyak) target updates
- Implement further DQN extensions: prioritized experience replay, dueling DQN, n-step returns